In [31]:
import pandas as pd
from pathlib import Path

RAW = Path("../data/raw/sources")

In [32]:
print(RAW.resolve())
print(RAW.exists())
sorted(p.name for p in RAW.iterdir())

/home/moussa-diaby/immo-analyse/data/raw/sources
True


['agence_teranga.xlsx',
 'import_historique.csv',
 'notaires.csv',
 'particuliers.json',
 'plateforme_immo_sn.csv']

In [33]:
plateforme = pd.read_csv(RAW / "plateforme_immo_sn.csv", sep=";", dtype=str)

print(plateforme.shape)
plateforme.head()

(875, 15)


,ref_plateforme,date_mise_en_ligne,type_transaction,categorie,region,commune,quartier,prix,devise,surface,pieces,sdb,description,statut,date_expiration
0,PLT-00001,2020-07-03,vente,Appartement,Tambacounda,Tambacounda,Nord,60581000,FCFA,173.3,1,1,Appartement à Tambacounda. Contact via platefo...,Vendu,2020-08-05
1,PLT-00002,2022-09-15,vente,Maison,Kolda,Kolda,Résidentiel,57.62 millions,FCFA,89.6,5,3,Maison à Kolda. Contact via plateforme.,Vendu,2022-11-19
2,PLT-00003,2019-01-15,vente,Terrain,Diourbel,Touba,Centre,9688000,FCFA,1457.3,0,0,Terrain à Touba. Contact via plateforme.,Vendu,2019-04-23
3,PLT-00004,2025-01-18,vente,Appartement,Thiès,Joal-Fadiouth,Sud,42273000,FCFA,69.8,3,2,Appartement à Joal-Fadiouth. Contact via plate...,En vente,2025-03-18
4,PLT-00005,2022-02-03,vente,Studio,Kaffrine,Kaffrine,Centre,26254000,FCFA,31.6,3,2,Studio à Kaffrine. Contact via plateforme.,Vendu,2022-03-26


Les valeurs manquantes
isna() ne détecte que les cases vraiment vides. Une case qui contient du texte comme "à négocier" joue le rôle d’un vide mais n’est pas comptée. On les cherchera à part, dans l’étude des formats.

In [34]:
# isna() renvoie True/False pour chaque case ; sum() compte les True par colonne
manquants = plateforme.isna().sum()

# Pourcentage = manquants / nombre total de lignes * 100, arrondi à 1 décimale
pourcentage = (manquants / len(plateforme) * 100).round(1)

# On assemble les deux dans un petit tableau, trié du plus incomplet au plus complet
resume = pd.DataFrame({"manquants": manquants, "pourcentage": pourcentage})
resume.sort_values("manquants", ascending=False)

,manquants,pourcentage
description,165,18.9
surface,5,0.6
ref_plateforme,0,0.0
categorie,0,0.0
region,0,0.0
date_mise_en_ligne,0,0.0
type_transaction,0,0.0
quartier,0,0.0
commune,0,0.0
devise,0,0.0


Les doublons

In [35]:
# duplicated() marque chaque ligne identique à une ligne PLUS HAUT dans le tableau
# (la première occurrence n'est pas comptée, seules les copies le sont)
doublons_exacts = plateforme.duplicated().sum()

# Même idée, mais uniquement sur la colonne identifiant
refs_dupliquees = plateforme["ref_plateforme"].duplicated().sum()

print("Lignes entièrement identiques :", doublons_exacts)
print("Références répétées           :", refs_dupliquees)

Lignes entièrement identiques : 15
Références répétées           : 15


Lignes concerrnées par les doublons

In [36]:
# keep=False marque TOUTES les lignes du groupe (l'original ET les copies),
# ce qui permet de les comparer côte à côte une fois triées par référence
plateforme[plateforme["ref_plateforme"].duplicated(keep=False)] \
    .sort_values("ref_plateforme") \
    .head(6)

,ref_plateforme,date_mise_en_ligne,type_transaction,categorie,region,commune,quartier,prix,devise,surface,pieces,sdb,description,statut,date_expiration
57,PLT-00058,2024-04-04,vente,Appartement,Diourbel,Mbacké,Sud,116544000,FCFA,160.2,5,2,Appartement à Mbacké. Contact via plateforme.,En vente,2024-06-27
864,PLT-00058,2024-04-04,vente,Appartement,Diourbel,Mbacké,Sud,116544000,FCFA,160.2,5,2,Appartement à Mbacké. Contact via plateforme.,En vente,2024-06-27
170,PLT-00171,2023-03-02,vente,Studio,Louga,Linguère,Populaire,41498000,FCFA,51.5,3,2,Studio à Linguère. Contact via plateforme.,Vendu,2023-06-18
869,PLT-00171,2023-03-02,vente,Studio,Louga,Linguère,Populaire,41498000,FCFA,51.5,3,2,Studio à Linguère. Contact via plateforme.,Vendu,2023-06-18
397,PLT-00398,2024-07-12,vente,Appartement,Fatick,Kaolack-Fatick,Résidentiel,31332000,FCFA,115.0,3,3,NaN,Vendu,2024-09-21
867,PLT-00398,2024-07-12,vente,Appartement,Fatick,Kaolack-Fatick,Résidentiel,31332000,FCFA,115.0,3,3,NaN,Vendu,2024-09-21


In [37]:
# POURQUOI : si une référence est répétée mais que la ligne n'est PAS identique,
# c'est un conflit (deux versions contradictoires), à traiter autrement qu'un doublon.
# On compare donc les deux comptages.

nb_refs_repetees = plateforme["ref_plateforme"].duplicated().sum()
nb_lignes_identiques = plateforme.duplicated().sum()

print("Références répétées  :", nb_refs_repetees)
print("Lignes identiques    :", nb_lignes_identiques)

# Si les deux nombres sont égaux, toute référence répétée est une copie exacte.
# S'ils diffèrent, la différence = nombre de conflits à examiner.
print("Conflits potentiels  :", nb_refs_repetees - nb_lignes_identiques)

Références répétées  : 15
Lignes identiques    : 15
Conflits potentiels  : 0


In [38]:
# Combien de ventes, de locations, de valeurs inattendues ?
# dropna=False : on compte aussi les cases vides au lieu de les cacher.
plateforme["type_transaction"].value_counts(dropna=False)
# La devise peut contenir une information en plus de la monnaie (une périodicité par exemple).
plateforme["devise"].value_counts(dropna=False)
# crosstab : une ligne par type de transaction, une colonne par devise.
# On voit si les deux colonnes sont liées.
pd.crosstab(plateforme["type_transaction"], plateforme["devise"], dropna=False)

devise,FCFA,FCFA/mois
type_transaction,,
location,0,177
vente,698,0


les formats du prix


In [39]:
def moule(serie: pd.Series) -> pd.Series:
    """Remplace chaque suite de chiffres par 9 et chaque mot par A, pour voir les formats."""
    return (
        serie.astype("string")                           # type texte qui accepte les valeurs manquantes
        .str.replace(r"\d+", "9", regex=True)            # une suite de chiffres -> un seul 9
        .str.replace(r"[A-Za-zÀ-ÿ]+", "A", regex=True)   # une suite de lettres (accents compris) -> un seul A
    )
# Combien de lignes pour chaque forme ? (dropna=False : on garde les valeurs manquantes)
moule(plateforme["prix"]).value_counts(dropna=False)

prix
9        804
9 9 9     50
9.9 A     15
A A        6
Name: count, dtype: Int64

Le « moule » remplace les chiffres par 9 et les mots par A, donc tous les prix écrits de la même façon ont le même moule, et on voit la liste des formats distincts.

In [40]:
# assign() ajoute une colonne "forme" sur une copie temporaire : plateforme n'est pas modifiée.
# groupby regroupe les lignes qui ont la même forme.
# size = nombre de lignes du groupe ; first = premier prix rencontré, pris comme exemple.
formes = (
    plateforme.assign(forme=moule(plateforme["prix"]))
    .groupby("forme", dropna=False)["prix"]
    .agg(nombre="size", exemple="first")
)
formes.sort_values("nombre", ascending=False)

,nombre,exemple
forme,,
9,804,60581000
9 9 9,50,438 746 000
9.9 A,15,57.62 millions
A A,6,à négocier


classer chaque famille
Forme	Classe	Raison	Règle
9	exploitable	uniquement des chiffres	conversion directe
9 9 9	normalisable	les espaces séparent les milliers, aucune autre lecture possible	enlever les espaces, puis convertir
9.9 A	normalisable	l’unité est écrite dans la valeur	nombre × 1 000 000
A A	inexploitable	aucun chiffre, on ne devine rien	quarantaine, motif « prix non numérique »

Le tableau n’affiche que la première valeur de chaque famille. Les 15 lignes 9.9 A contiennent-elles toutes « millions » ? Pourraient-elles contenir « milliards » ou « M » ? Les 6 A A sont-elles toutes « à négocier » ? On vérifie avant d’écrire une règle, sinon une règle sûre pour l’exemple peut être fausse pour le reste.

In [41]:
# POURQUOI : le groupby n'a montré qu'un exemple par famille. Ici on regarde TOUT le groupe.
formes_prix = moule(plateforme["prix"])

# Quels mots apparaissent dans la famille "9.9 A" ? extract(...) récupère la partie lettres.
mots = plateforme.loc[formes_prix == "9.9 A", "prix"].str.extract(r"([A-Za-zÀ-ÿ]+)")[0]
print(mots.value_counts())

# Quelles valeurs exactes dans la famille "A A" ?
print(plateforme.loc[formes_prix == "A A", "prix"].value_counts())

0
millions    15
Name: count, dtype: int64
prix
à négocier    6
Name: count, dtype: int64


c'est bon c'est confirmé

on va passer a l'étape des conversions 

le cahier impose l’indicateur « erreurs de conversion par champ ». Pour le mesurer, on tente la conversion sans rien corriger et on compte ce qui résiste

si on convertit directement prix sans rien corriger, quelles familles échouent, et combien de lignes en tout ?


le cahier insiste pour distinguer ventes et locations avant de normaliser un prix. Ici, on veut savoir si les familles « particulières » (espaces, millions, à négocier) se trouvent dans les ventes, les locations, ou les deux. Ça influence la façon dont on documentera les décisions.

In [42]:
# errors="coerce" : une valeur non convertible devient NaN au lieu de provoquer une erreur.
prix_num = pd.to_numeric(plateforme["prix"], errors="coerce")

# Échec = la case contenait quelque chose (notna) mais la conversion a donné NaN.
echecs = prix_num.isna() & plateforme["prix"].notna()

print("Échecs de conversion :", echecs.sum())
print("Conversions réussies :", prix_num.notna().sum())
# crosstab : une ligne par forme de prix, une colonne par type de transaction.
# Chaque case = nombre de lignes ayant cette forme ET ce type de transaction.
pd.crosstab(moule(plateforme["prix"]), plateforme["type_transaction"])

Échecs de conversion : 71
Conversions réussies : 804


type_transaction,location,vente
prix,,
9,177,627
9 9 9,0,50
9.9 A,0,15
A A,0,6


Le croisement apporte deux informations :

Constat	Conséquence
Les 177 locations sont toutes dans la famille 9	les loyers sont écrits proprement, aucune normalisation nécessaire
Les 71 prix problématiques sont tous des ventes

Bilan après normalisation prévue : sur 71 échecs, 65 sont corrigeables (50 espaces + 15 millions) et 6 restent inexploitables. On passe de 804 à 869 prix utilisables sur 875.

La fonction normaliser_prix

Pourquoi une fonction qui travaille sur UNE valeur : plus facile à comprendre et à tester. On l’appliquera ensuite à toute la colonne. Elle renvoie aussi un statut avec la valeur, car le rapport doit distinguer ce qui était conforme, ce qui a été corrigé et ce qui est rejeté (indicateurs obligatoires).

In [43]:
import re

def normaliser_prix(texte):
    """Convertit un prix écrit en texte en nombre (FCFA), sans jamais deviner.

    Renvoie un couple (valeur, statut) :
      (nombre, "conforme")     : déjà des chiffres seuls
      (nombre, "corrige")      : converti grâce à une règle sûre
      (None, "non_numerique")  : aucun nombre fiable -> quarantaine
      (None, "manquant")       : case vide
    """
    # Case vide : rien à convertir
    if texte is None or pd.isna(texte):
        return None, "manquant"

    brut = str(texte).strip()   # strip() enlève les espaces en début et fin

    # Règle 1 : chiffres seuls -> conversion directe
    if re.fullmatch(r"\d+", brut):
        return float(brut), "conforme"

    # Règle 2 : groupes de 3 chiffres séparés par des espaces ("438 746 000").
    # \s accepte aussi les espaces insécables, invisibles à l'œil mais fréquents dans les exports.
    if re.fullmatch(r"\d{1,3}(\s\d{3})+", brut):
        return float(re.sub(r"\s", "", brut)), "corrige"

    # Règle 3 : nombre décimal + le mot "millions" ("57.62 millions")
    m = re.fullmatch(r"(\d+(?:\.\d+)?)\s*millions?", brut, flags=re.IGNORECASE)
    if m:
        # round() : en informatique, 57.62 * 1_000_000 peut donner 57619999.99999999
        return float(round(float(m.group(1)) * 1_000_000)), "corrige"

    # Tout le reste ("à négocier", ...) : on ne devine rien
    return None, "non_numerique"

Test de la fonction avant de l’utiliser

Une fonction non testée est une supposition. On teste trois sortes de cas : normaux, invalides et limites.

In [44]:
# assert : si la condition est fausse, Python lève une erreur et montre laquelle.
# Cas normaux
assert normaliser_prix("60581000") == (60581000.0, "conforme")
assert normaliser_prix("438 746 000") == (438746000.0, "corrige")
assert normaliser_prix("57.62 millions") == (57620000.0, "corrige")

# Cas invalides
assert normaliser_prix("à négocier") == (None, "non_numerique")
assert normaliser_prix(None) == (None, "manquant")

# Cas limites : on ne doit PAS corriger ces valeurs ambiguës
assert normaliser_prix("12 34") == (None, "non_numerique")     # ce ne sont pas des milliers
assert normaliser_prix("1,250") == (None, "non_numerique")     # virgule : ambigu, on ne devine pas
assert normaliser_prix("-5000") == (None, "non_numerique")     # le signe moins n'est pas géré

# Un prix de 0 est bien ÉCRIT (chiffres seuls), donc la fonction l'accepte : conforme au FORMAT.
# Le juger absurde (un bien à 0 FCFA) est le rôle d'une autre étape : la plausibilité.
assert normaliser_prix("0") == (0.0, "conforme")

# Espace insécable (invisible à l'œil, fréquent dans les exports) : \s le reconnaît aussi
assert normaliser_prix("438\u00a0746\u00a0000") == (438746000.0, "corrige")

# Casse du mot : "Millions" doit marcher aussi, grâce à re.IGNORECASE
assert normaliser_prix("57.62 Millions") == (57620000.0, "corrige")

print("Tous les tests passent")

Tous les tests passent


### Application de normaliser_prix à la colonne prix

In [45]:
# apply() appelle normaliser_prix sur chaque valeur de la colonne "prix"
resultats = plateforme["prix"].apply(normaliser_prix)

# Chaque résultat est un couple (valeur, statut) : on le sépare en deux colonnes, dans un
# NOUVEAU tableau. plateforme (le brut) n'est pas modifiée.
prix_norm = pd.DataFrame(
    resultats.tolist(),
    columns=["prix_num", "prix_statut"],
    index=plateforme.index,   # même index que plateforme : chaque ligne reste alignée avec son origine
)

prix_norm["prix_statut"].value_counts()

prix_statut
conforme         804
corrige           65
non_numerique      6
Name: count, dtype: int64